# ISM Project Shafe — Report Figure GeneratorThis notebook regenerates publication/report figures from already-preservedexperiment artifacts.**It performs:**- NO model training- NO threshold tuning- NO TEST optimization- NO modification of frozen scientific artifactsAll figures appear directly in notebook output cells.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────# Edit this cell to change appearance or behavior.GITHUB_REPO = "https://github.com/MH-Shafe/ISM-project.git"REPO_DIR    = "ISM-project"AUTO_CLONE  = True       # clone from GitHub if repo not found locallySAVE_FIGURES = False     # True → save PNGs; False → display inline onlyDPI         = 150FIGSIZE     = (10, 6)TITLE_SIZE  = 15LABEL_SIZE  = 12TICK_SIZE   = 10LEGEND_SIZE = 10LINE_WIDTH  = 2.0BAR_WIDTH   = 0.22GRID_ALPHA  = 0.20

In [ ]:
import os, subprocessfrom pathlib import Pathdef find_repo():    """Locate the ISM-project repository root."""    cwd = Path.cwd()    # 1. Check cwd itself    if (cwd / "scripts").is_dir() and (cwd / "reports").is_dir():        return cwd    # 2. Check ./ISM-project    if (cwd / REPO_DIR).is_dir():        return cwd / REPO_DIR    # 3. Check /kaggle/working/ISM-project    kaggle_path = Path("/kaggle/working") / REPO_DIR    if kaggle_path.is_dir():        return kaggle_path    # 4. Check parent directories    for p in [cwd.parent, cwd.parent.parent]:        if (p / "scripts").is_dir() and (p / "reports").is_dir():            return p    return NoneREPO_ROOT = find_repo()if REPO_ROOT is None and AUTO_CLONE:    print("Repository not found locally. Cloning from GitHub...")    try:        subprocess.check_call(            ["git", "clone", GITHUB_REPO, REPO_DIR],            cwd=str(Path.cwd()),            timeout=60,        )        REPO_ROOT = Path.cwd() / REPO_DIR        print(f"Cloned to {REPO_ROOT}")    except Exception as e:        print(f"Clone failed: {e}")        print("Enable Internet in Kaggle, or upload/attach the repository.")        REPO_ROOT = Noneif REPO_ROOT is not None:    print(f"Repository root: {REPO_ROOT}")    try:        sha_before = subprocess.check_output(            ["git", "rev-parse", "--short", "HEAD"],            cwd=str(REPO_ROOT), text=True,        ).strip()        print(f"Git commit before update: {sha_before}")    except Exception:        sha_before = None        print("Git commit before update: (unknown)")    # Attempt safe pull if worktree is clean    try:        status = subprocess.check_output(            ["git", "status", "--porcelain"],            cwd=str(REPO_ROOT), text=True,        ).strip()        if not status:            subprocess.check_call(                ["git", "fetch", "origin", "main"],                cwd=str(REPO_ROOT), timeout=30,            )            subprocess.check_call(                ["git", "pull", "--ff-only", "origin", "main"],                cwd=str(REPO_ROOT), timeout=30,            )            sha_after = subprocess.check_output(                ["git", "rev-parse", "--short", "HEAD"],                cwd=str(REPO_ROOT), text=True,            ).strip()            print(f"Git commit after update:  {sha_after}")        else:            print("Repository has local changes; skipping automatic pull.")    except Exception as e:        print(f"Git update failed: {e}")else:    print("ERROR: Repository not found. Cannot continue.")

In [ ]:
%matplotlib inlinefrom pathlib import Pathimport json, hashlib, warningsimport numpy as npimport pandas as pdimport matplotlibimport matplotlib.pyplot as pltfrom IPython.display import displayfrom sklearn.metrics import (    roc_curve, roc_auc_score,    precision_recall_curve, average_precision_score,    confusion_matrix,)warnings.filterwarnings("ignore", category=UserWarning)plt.rcParams.update({    "figure.dpi": DPI,    "font.size": TICK_SIZE,    "axes.titlesize": TITLE_SIZE,    "axes.labelsize": LABEL_SIZE,    "legend.fontsize": LEGEND_SIZE,})def show_figure(fig, filename=None):    """Display figure inline and optionally save it."""    if SAVE_FIGURES and filename:        out = REPO_ROOT / "reports" / "figures" / "notebook_generated"        out.mkdir(parents=True, exist_ok=True)        fig.savefig(out / filename, dpi=300, bbox_inches="tight")        print(f"Saved: {out / filename}")    display(fig)    plt.close(fig)

In [ ]:
def first_existing(*paths):    """Return the first path that exists, or None."""    for p in paths:        if p.exists():            return p    return Noneablation_path = first_existing(    REPO_ROOT / "reports/final/END_TO_END_ABLATION_SUMMARY.csv",    REPO_ROOT / "reports/final/END_TO_END_ABLATION_SUMMARY.json",)shap_path = first_existing(    REPO_ROOT / "reports/artifacts/phase11_global_importance.csv",    REPO_ROOT / "artifacts/summaries/phase11_global_importance.json",)baseline_csv_path = first_existing(    REPO_ROOT / "reports/final/BASELINE_MODEL_COMPARISON.csv",    REPO_ROOT / "reports/artifacts/baseline_benchmark/BASELINE_MODEL_COMPARISON.csv",)artifacts = {    "ablation":              ablation_path,    "baseline_csv":          baseline_csv_path,    "all_results":           REPO_ROOT / "reports/artifacts/baseline_benchmark/all_results.json",    "freeze":                REPO_ROOT / "reports/artifacts/baseline_benchmark/BENCHMARK_FREEZE.json",    "bootstrap_ci":          REPO_ROOT / "reports/artifacts/baseline_benchmark/bootstrap_confidence_intervals.json",    "pred_dir":              REPO_ROOT / "reports/artifacts/baseline_benchmark/predictions",    "shap":                  shap_path,    "decisions_parquet":     first_existing(        REPO_ROOT / "demo_artifacts/final_user_day_decisions.parquet",        REPO_ROOT / "reports/artifacts/phase20/final_user_day_decisions.parquet",    ),}rows = []for name, path in artifacts.items():    exists = path is not None and path.exists() if isinstance(path, Path) else False    display = str(path.relative_to(REPO_ROOT)) if exists and path is not None else str(path)    rows.append({"Artifact": name, "Path": display, "Exists": "YES" if exists else "NO"})pd.DataFrame(rows)

In [ ]:
PRED_DIR = artifacts["pred_dir"]with open(artifacts["all_results"]) as f:    all_results = json.load(f)MODEL_MAP = {    "logistic":         ("Logistic Regression", "logistic_test_predictions.parquet"),    "random_forest":    ("Random Forest",       "random_forest_test_predictions.parquet"),    "xgboost":          ("XGBoost",             "xgboost_test_predictions.parquet"),    "catboost":         ("CatBoost",            "catboost_test_predictions.parquet"),    "lightgbm":         ("LightGBM",            "lightgbm_benchmark_test_predictions.parquet"),}EXPECTED = {    "logistic":      {"roc_auc": 0.882866, "pr_auc": 0.009934},    "random_forest": {"roc_auc": 0.772914, "pr_auc": 0.183094},    "xgboost":       {"roc_auc": 0.940020, "pr_auc": 0.269326},    "catboost":      {"roc_auc": 0.919902, "pr_auc": 0.179216},    "lightgbm":      {"roc_auc": 0.939157, "pr_auc": 0.267776},}pred_data = {}  # model_key -> {"y": array, "score": array, "df": DataFrame}val_rows = []for key, (display_name, fname) in MODEL_MAP.items():    fpath = PRED_DIR / fname    if not fpath.exists():        val_rows.append({"Model": display_name, "Status": "MISSING"})        continue    df = pd.read_parquet(fpath)    y = df["malicious"].astype(int).to_numpy()    s = df["score"].astype(float).to_numpy()    roc = roc_auc_score(y, s)    pr  = average_precision_score(y, s)    exp = EXPECTED[key]    roc_ok = abs(roc - exp["roc_auc"]) < 0.001    pr_ok  = abs(pr  - exp["pr_auc"])  < 0.001    status = "PASS" if (roc_ok and pr_ok) else "FAIL"    pred_data[key] = {"y": y, "score": s, "df": df, "display": display_name}    val_rows.append({        "Model": display_name, "Rows": len(df), "Positives": int(y.sum()),        "ROC-AUC": f"{roc:.6f}", "PR-AUC": f"{pr:.6f}", "Status": status,    })val_df = pd.DataFrame(val_rows)print(f"Models loaded: {len(pred_data)}/5")print(f"All PASS: {(val_df['Status'] == 'PASS').all()}")val_df

## Figure 1 — Final System Architecture

In [ ]:
fig, ax = plt.subplots(figsize=(9, 11))ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")steps = [    "CERT r4.2",    "Validation & Preprocessing",    "Leakage-Safe User-Day Aggregation",    "Behavioral + Graph Features (12)",    "Frozen LightGBM (lgbm-graph-v1)",    "Frozen Decision Policy (t=0.9186)",    "Conformal Uncertainty",    "SHAP Explanations",    "Trust / Context Diagnostics",    "Deterministic Decision Layer",    "Analyst-Facing User-Day Output",]ys = np.linspace(0.94, 0.08, len(steps))for i, (label, y) in enumerate(zip(steps, ys)):    ax.text(0.5, y, label, ha="center", va="center", fontsize=12,            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black", lw=1.2))    if i < len(steps) - 1:        ax.annotate("", xy=(0.5, ys[i+1]+0.026), xytext=(0.5, y-0.026),                    arrowprops=dict(arrowstyle="->", lw=1.2))ax.text(0.5, 0.985, "Figure 1. Final System Architecture",        ha="center", va="top", fontsize=TITLE_SIZE, fontweight="bold")ax.text(0.5, 0.025,        "Adaptive Risk was evaluated and rejected; it is not part of the production path.",        ha="center", va="bottom", fontsize=9, style="italic")show_figure(fig, "figure_01_system_architecture.png")

## Figure 2 — Leakage-Safe Chronological Split

In [ ]:
with open(artifacts["ablation"]) as f:    abl_splits = json.load(f).get("splits", {})sp_train = abl_splits.get("train", {})sp_cal   = abl_splits.get("cal", {})sp_test  = abl_splits.get("test", {})rows_n = [sp_train.get("rows", 0), sp_cal.get("rows", 0), sp_test.get("rows", 0)]pos_n  = [sp_train.get("positives", 0), sp_cal.get("positives", 0), sp_test.get("positives", 0)]names  = ["TRAIN", "CAL", "TEST"]dates  = ["≤ 2011-01-31", "2011-02-01 → 2011-03-31", "≥ 2011-04-01"]fig, ax = plt.subplots(figsize=FIGSIZE)x = np.arange(len(names))bars = ax.bar(x, rows_n, color=["#4C72B0", "#55A868", "#C44E52"])ax.set_xticks(x); ax.set_xticklabels(names, fontsize=TICK_SIZE)ax.set_ylabel("User-Days", fontsize=LABEL_SIZE)ax.set_title("Figure 2. Leakage-Safe Chronological Data Split",             fontsize=TITLE_SIZE, fontweight="bold")ax.grid(axis="y", alpha=GRID_ALPHA)ymax = max(rows_n)for bar, n, p, d in zip(bars, rows_n, pos_n, dates):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ymax*0.025,            f"{d}\n{n:,} rows\n{p:,} positive",            ha="center", va="bottom", fontsize=9)ax.text(0.5, -0.18,        "TRAIN → CAL → TEST  |  No temporal overlap  |  TEST never used for training or threshold fitting",        transform=ax.transAxes, ha="center", fontsize=9)fig.subplots_adjust(bottom=0.22, top=0.90)show_figure(fig, "figure_02_chronological_split.png")

## Figure 3 — CERT r4.2 Class Distribution

In [ ]:
total      = sum(rows_n)malicious  = sum(pos_n)benign     = total - maliciousfig, ax = plt.subplots(figsize=FIGSIZE)bars = ax.bar(["Benign", "Malicious"], [benign, malicious],              color=["#4C72B0", "#C44E52"])ax.set_ylabel("User-Day Count", fontsize=LABEL_SIZE)ax.set_title("Figure 3. CERT r4.2 User-Day Class Distribution",             fontsize=TITLE_SIZE, fontweight="bold")ax.grid(axis="y", alpha=GRID_ALPHA)for bar, val in zip(bars, [benign, malicious]):    pct = 100.0 * val / total    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,            f"{val:,}\n({pct:.2f}%)", ha="center", va="bottom", fontsize=10)ax.text(0.5, -0.12, f"Total = {total:,} user-days  |  Malicious = {malicious:,}",        transform=ax.transAxes, ha="center", fontsize=9)fig.subplots_adjust(bottom=0.16, top=0.92)show_figure(fig, "figure_03_class_distribution.png")

## Figure 4 — Leakage-Safe Ablation Comparison

In [ ]:
abl_path = artifacts["ablation"]if abl_path is None:    raise FileNotFoundError("Ablation artifact not found")if abl_path.suffix == ".csv":    abl_raw = pd.read_csv(abl_path)    abl_raw = abl_raw[abl_raw.get("table", pd.Series(["table1"]*len(abl_raw))) == "table1"].head(3)    models_abl = abl_raw["label"].tolist()    pr_auc     = abl_raw["auc_pr"].astype(float).tolist()    f1_vals    = abl_raw["f1"].astype(float).tolist()    mcc_vals   = abl_raw["mcc"].astype(float).tolist()else:    with open(abl_path) as f:        abl_json = json.load(f)    table1 = abl_json.get("table1_directly_comparable", [])    models_abl, pr_auc, f1_vals, mcc_vals = [], [], [], []    for entry in table1[:3]:        models_abl.append(entry["label"])        m = entry.get("metrics", {})        pr_auc.append(float(m.get("auc_pr", 0)))        f1_vals.append(float(m.get("f1", 0)))        mcc_vals.append(float(m.get("mcc", 0)))x = np.arange(len(models_abl))w = BAR_WIDTHfig, ax = plt.subplots(figsize=FIGSIZE)ax.bar(x - w, pr_auc,  w, label="PR-AUC", color="#4C72B0")ax.bar(x,     f1_vals, w, label="F1",     color="#55A868")ax.bar(x + w, mcc_vals,w, label="MCC",    color="#C44E52")ax.set_xticks(x)ax.set_xticklabels(["Behavioral\nonly", "Graph\nonly", "Behavioral\n+ Graph"],                    fontsize=TICK_SIZE)ax.set_ylabel("Score", fontsize=LABEL_SIZE)ax.set_ylim(bottom=0)ax.set_title("Figure 4. Leakage-Safe Ablation Comparison (Arms A / B / C)",             fontsize=TITLE_SIZE, fontweight="bold")ax.legend(fontsize=LEGEND_SIZE)ax.grid(axis="y", alpha=GRID_ALPHA)for i, (p, f, m) in enumerate(zip(pr_auc, f1_vals, mcc_vals)):    ax.text(i - w, p + 0.005, f"{p:.3f}", ha="center", va="bottom", fontsize=8)    ax.text(i,     f + 0.005, f"{f:.3f}", ha="center", va="bottom", fontsize=8)    ax.text(i + w, m + 0.005, f"{m:.3f}", ha="center", va="bottom", fontsize=8)fig.subplots_adjust(bottom=0.20, top=0.90)show_figure(fig, "figure_04_ablation.png")

## Figure 5A — ROC Curves on Chronological TEST

In [ ]:
ORDER = ["logistic", "random_forest", "xgboost", "catboost", "lightgbm"]COLORS = {"logistic": "#C44E52", "random_forest": "#DD8452",          "xgboost": "#55A868", "catboost": "#4C72B0", "lightgbm": "#937860"}fig, ax = plt.subplots(figsize=FIGSIZE)for key in ORDER:    if key not in pred_data:        continue    d = pred_data[key]    fpr, tpr, _ = roc_curve(d["y"], d["score"])    auc_val = roc_auc_score(d["y"], d["score"])    ax.plot(fpr, tpr, lw=LINE_WIDTH, color=COLORS[key],            label=f'{d["display"]} (AUC={auc_val:.3f})')ax.plot([0, 1], [0, 1], ls="--", lw=1, color="grey", label="Random")ax.set_xlabel("False Positive Rate", fontsize=LABEL_SIZE)ax.set_ylabel("True Positive Rate", fontsize=LABEL_SIZE)ax.set_title("Figure 5A. ROC Curves — All Five Models on Chronological TEST",             fontsize=TITLE_SIZE, fontweight="bold")ax.legend(fontsize=LEGEND_SIZE, loc="lower right")ax.grid(alpha=GRID_ALPHA)show_figure(fig, "figure_05a_roc.png")

## Figure 5B — Precision–Recall Curves on Chronological TEST

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)for key in ORDER:    if key not in pred_data:        continue    d = pred_data[key]    prec, rec, _ = precision_recall_curve(d["y"], d["score"])    ap = average_precision_score(d["y"], d["score"])    ax.plot(rec, prec, lw=LINE_WIDTH, color=COLORS[key],            label=f'{d["display"]} (AP={ap:.3f})')prevalence = 30 / 47000ax.axhline(y=prevalence, ls=":", lw=1, color="grey",           label=f"Random / prevalence ({prevalence:.4f})")ax.set_xlabel("Recall", fontsize=LABEL_SIZE)ax.set_ylabel("Precision", fontsize=LABEL_SIZE)ax.set_title("Figure 5B. Precision–Recall Curves — All Five Models",             fontsize=TITLE_SIZE, fontweight="bold")ax.legend(fontsize=LEGEND_SIZE, loc="upper right")ax.grid(alpha=GRID_ALPHA)show_figure(fig, "figure_05b_pr.png")

## Figure 6 — Global SHAP Feature Importance

In [ ]:
shap_path = artifacts["shap"]if shap_path is None:    raise FileNotFoundError("SHAP importance artifact not found")if shap_path.suffix == ".csv":    shap_df = pd.read_csv(shap_path)    shap_df = shap_df.rename(columns={"mean_abs_shap": "importance"})else:    with open(shap_path) as f:        shap_json = json.load(f)    per_feat = shap_json.get("calibration", shap_json).get("importance", shap_json).get("per_feature", {})    rows = []    for feat, vals in per_feat.items():        rows.append({"feature": feat, "importance": vals.get("mean_abs", 0)})    shap_df = pd.DataFrame(rows)shap_df = shap_df.sort_values("importance", ascending=True)fig_h = max(6, 0.42 * len(shap_df) + 2)fig, ax = plt.subplots(figsize=(10, fig_h))bars = ax.barh(shap_df["feature"], shap_df["importance"], color="#4C72B0")ax.set_xlabel("Mean Absolute SHAP Contribution", fontsize=LABEL_SIZE)ax.set_title("Figure 6. Global SHAP Feature Importance (Phase 11, 12 features)",             fontsize=TITLE_SIZE, fontweight="bold")ax.grid(axis="x", alpha=GRID_ALPHA)for bar, val in zip(bars, shap_df["importance"]):    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,            f"{val:.3f}", va="center", fontsize=9)fig.subplots_adjust(left=0.32, top=0.94)show_figure(fig, "figure_06_shap.png")

## Figure 7 — Final Operational Decision Distribution

In [ ]:
ddf = pd.read_parquet(artifacts["decisions_parquet"])counts = ddf["risk_level"].astype(str).str.upper().value_counts()preferred = ["ALERT", "BORDERLINE", "MONITOR", "NON-ALERT"]vals  = [int(counts.get(x, 0)) for x in preferred]total_d = sum(vals)fig, ax = plt.subplots(figsize=FIGSIZE)bars = ax.bar(preferred, vals, color=["#C44E52", "#DD8452", "#55A868", "#4C72B0"])ax.set_ylabel("User-Day Count", fontsize=LABEL_SIZE)ax.set_title("Figure 7. Final Operational Decision Distribution (Phase 20)",             fontsize=TITLE_SIZE, fontweight="bold")ax.grid(axis="y", alpha=GRID_ALPHA)ymax = max(vals)for bar, v in zip(bars, vals):    pct = 100.0 * v / total_d    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ymax*0.015,            f"{v:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=9)ax.text(0.5, -0.13, f"Complete Phase 20 population: {total_d:,} user-days",        transform=ax.transAxes, ha="center", fontsize=9)fig.subplots_adjust(bottom=0.18, top=0.92)show_figure(fig, "figure_07_decisions.png")

## Figure 8 — Post-Freeze Baseline Comparison

In [ ]:
if artifacts["baseline_csv"] is not None and artifacts["baseline_csv"].exists():    bdf = pd.read_csv(artifacts["baseline_csv"])    print("Figure 8 source: BASELINE_MODEL_COMPARISON.csv")else:    print("Figure 8 source: all_results.json fallback")    required_models = ["Logistic Regression", "Random Forest", "XGBoost", "CatBoost", "LightGBM (frozen)"]    rows = []    for model in required_models:        r = all_results.get(model, {})        rows.append({            "Model": model,            "PR-AUC": r.get("pr_auc", 0),            "F1": r.get("f1", 0),            "MCC": r.get("mcc", 0),        })    bdf = pd.DataFrame(rows)MODEL_ORDER = ["Logistic Regression", "Random Forest", "XGBoost", "CatBoost", "LightGBM (frozen)"]name_col = "Model"if "PR-AUC" not in bdf.columns and "pr_auc" in bdf.columns:    bdf = bdf.rename(columns={"pr_auc": "PR-AUC", "f1": "F1", "mcc": "MCC"})kept = bdf[bdf[name_col].isin(MODEL_ORDER)].copy()kept["_order"] = kept[name_col].map({m: i for i, m in enumerate(MODEL_ORDER)})kept = kept.sort_values("_order").drop(columns=["_order"]).reset_index(drop=True)DISPLAY = {    "Logistic Regression": "Logistic\nRegression",    "Random Forest":       "Random\nForest",    "XGBoost":             "XGBoost",    "CatBoost":            "CatBoost",    "LightGBM (frozen)":   "LightGBM\n(frozen)",}metrics = ["PR-AUC", "F1", "MCC"]mcolors = ["#4C72B0", "#55A868", "#C44E52"]x = np.arange(len(kept))w = BAR_WIDTHfig, ax = plt.subplots(figsize=(12, 6.5))for i, (metric, color) in enumerate(zip(metrics, mcolors)):    vals = pd.to_numeric(kept[metric], errors="coerce").to_numpy()    bars = ax.bar(x + (i-1)*w, vals, w, label=metric, color=color)    for bar, v in zip(bars, vals):        if np.isfinite(v):            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.006,                    f"{v:.3f}", ha="center", va="bottom", fontsize=8, rotation=90)ax.set_xticks(x)ax.set_xticklabels([DISPLAY.get(m, m) for m in kept[name_col]], fontsize=TICK_SIZE)ax.set_ylabel("Score", fontsize=LABEL_SIZE)ax.set_ylim(0, 0.45)ax.set_title("Figure 8. Post-Freeze Baseline Model Comparison",             fontsize=TITLE_SIZE, fontweight="bold")ax.legend(fontsize=LEGEND_SIZE)ax.grid(axis="y", alpha=GRID_ALPHA)ax.text(0.5, -0.18, "Same 12 features and chronological split; thresholds selected on CAL only.",        transform=ax.transAxes, ha="center", fontsize=9)fig.subplots_adjust(bottom=0.22, top=0.92)show_figure(fig, "figure_08_baselines.png")

## Figure 9 — TEST Confusion Matrices

In [ ]:
with open(artifacts["all_results"]) as f:    results = json.load(f)CM_ORDER = [    ("Logistic Regression", "logistic"),    ("Random Forest",       "random_forest"),    ("XGBoost",             "xgboost"),    ("CatBoost",            "catboost"),    ("LightGBM (frozen)",   "lightgbm"),]for display_name, key in CM_ORDER:    if key not in pred_data:        print(f"Skipping {display_name}: predictions not loaded")        continue    d = pred_data[key]    threshold = results[display_name]["threshold"]    y_pred = (d["score"] >= threshold).astype(int)    tn, fp, fn, tp = confusion_matrix(d["y"], y_pred, labels=[0,1]).ravel()    cm = np.array([[tn, fp], [fn, tp]])    fig, ax = plt.subplots(figsize=(5, 4.5))    im = ax.imshow(cm, cmap="Blues", aspect="auto")    for i in range(2):        for j in range(2):            val = cm[i, j]            color = "white" if val > cm.max() / 2 else "black"            ax.text(j, i, f"{val:,}", ha="center", va="center",                    fontsize=14, fontweight="bold", color=color)    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])    ax.set_xticklabels(["Predicted\nNegative", "Predicted\nPositive"], fontsize=10)    ax.set_yticklabels(["Actual\nNegative", "Actual\nPositive"], fontsize=10)    ax.set_title(f"Confusion Matrix — {display_name}\n(threshold={threshold:.4f})",                 fontsize=12, fontweight="bold")    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)    fig.tight_layout()    show_figure(fig, f"figure_09_confusion_{key}.png")

## Supplementary Figure — Bootstrap 95% Confidence Intervals

In [ ]:
with open(artifacts["bootstrap_ci"]) as f:    ci = json.load(f)ci_models = ci if "Logistic Regression" in ci else ci.get("models", {})ci_keys = ["Logistic Regression", "Random Forest", "XGBoost", "CatBoost", "LightGBM (benchmark)"]ci_keys = [k for k in ci_keys if k in ci_models]metrics_ci = ["roc_auc", "pr_auc", "f1", "mcc"]metric_labels = ["ROC-AUC", "PR-AUC", "F1", "MCC"]fig, axes = plt.subplots(2, 2, figsize=(14, 10))axes = axes.flatten()for ax, metric, label in zip(axes, metrics_ci, metric_labels):    points, lows, highs, names = [], [], [], []    for mname in ci_keys:        mdata = ci_models[mname].get(metric, {})        if not mdata:            continue        points.append(mdata.get("mean", mdata.get("point_estimate", 0)))        lows.append(mdata.get("ci_95_lower", mdata.get("ci_lower", 0)))        highs.append(mdata.get("ci_95_upper", mdata.get("ci_upper", 0)))        names.append(mname.replace(" ", "\n") if len(mname) > 12 else mname)    x = np.arange(len(names))    yerr = np.array([np.array(points) - np.array(lows), np.array(highs) - np.array(points)])    ax.errorbar(x, points, yerr=yerr, fmt="o", capsize=5, lw=1.5, color="#4C72B0")    ax.set_xticks(x)    ax.set_xticklabels(names, fontsize=8, rotation=15, ha="right")    ax.set_ylabel(label, fontsize=LABEL_SIZE)    ax.set_title(f"{label} — 95% Bootstrap CI (1000 resamples)",                 fontsize=11, fontweight="bold")    ax.grid(axis="y", alpha=GRID_ALPHA)fig.suptitle("Supplementary Figure. Bootstrap Confidence Intervals",             fontsize=TITLE_SIZE, fontweight="bold", y=1.01)fig.tight_layout()show_figure(fig, "figure_10_bootstrap_ci.png")

## Final Validation Summary

In [ ]:
baseline_source_ok = (    (artifacts["baseline_csv"] is not None and artifacts["baseline_csv"].exists())    or    (artifacts["all_results"].exists() and all(        m in all_results for m in ["Logistic Regression", "Random Forest", "XGBoost", "CatBoost"]    )))checks = {    "Repository detected":              REPO_ROOT is not None,    "Ablation artifact":                artifacts["ablation"] is not None and artifacts["ablation"].exists(),    "Baseline comparison source":       baseline_source_ok,    "All results JSON":                 artifacts["all_results"].exists(),    "Freeze JSON":                      artifacts["freeze"].exists(),    "Bootstrap CI JSON":                artifacts["bootstrap_ci"].exists(),    "SHAP artifact":                    artifacts["shap"] is not None and artifacts["shap"].exists(),    "Decisions parquet":                artifacts["decisions_parquet"] is not None and artifacts["decisions_parquet"].exists(),    "Prediction dir":                   artifacts["pred_dir"].is_dir(),    "All 5 models loaded":              len(pred_data) == 5,    "All predictions PASS":             (val_df["Status"] == "PASS").all() if len(val_df) == 5 else False,    "SAVE_FIGURES default":             SAVE_FIGURES == False,}summary = pd.DataFrame([    {"Check": k, "Result": "PASS" if v else "FAIL"}    for k, v in checks.items()])print(summary.to_string(index=False))print(f"\nOverall: {'ALL PASS' if all(checks.values()) else 'SOME FAILED'}")